# LinguaFranca — Phase 1: Dataset Construction
> **Do Internal Failures Predict External Ones?**

This notebook runs the full Phase 1 pipeline:
1. Install dependencies
2. Unpack the source code
3. Download 2WikiMultihopQA + HotpotQA
4. Generate XML-tagged CoT trajectories (Qwen2.5-3B-Instruct)
5. Label each hop as success/failure
6. Build counterfactual examples
7. Write `train/val/test.jsonl` splits

**GPU required** — set accelerator to GPU (T4 x2 or P100) in Notebook Settings.

## Step 0 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — enable GPU in Settings!')

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install nnsight sentence-transformers SPARQLWrapper datasets accelerate -q

## Step 2 — Unpack Source Code

In [ ]:
import zipfile, os, sys

# The zip was uploaded as a Kaggle Dataset
ZIP_PATH = '/kaggle/input/linguafranca-src/linguafranca_src.zip'
WORK_DIR = '/kaggle/working/LinguaFranca'

os.makedirs(WORK_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(WORK_DIR)

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

## Step 3 — Configure for Kaggle (Qwen, paths)

In [ ]:
import yaml

with open('configs/data_config.yaml') as f:
    cfg = yaml.safe_load(f)

# ── Switch to Qwen (no HF token needed on Kaggle) ────────────
cfg['model']['name']               = 'Qwen/Qwen2.5-3B-Instruct'
cfg['model']['trust_remote_code']  = True
cfg['model']['dtype']              = 'bfloat16'   # better on A100/T4

# ── Point data dirs to /kaggle/working ───────────────────────
cfg['data']['raw_dir']             = '/kaggle/working/data/raw'
cfg['data']['processed_dir']       = '/kaggle/working/data/processed'
cfg['data']['hidden_states_dir']   = '/kaggle/working/data/hidden_states'
cfg['matching']['wikidata_aliases_path'] = '/kaggle/working/data/raw/wikidata_aliases.json'

# ── Reduce scale for a ~4-hour Kaggle run ────────────────────
# Full 8000 examples takes ~8 hrs on T4. Start with 2000 to test.
cfg['data']['n_source_examples']   = 2000   # increase to 8000 for final run
cfg['generation']['batch_size']    = 4

# Save modified config
with open('configs/data_config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config updated:')
print(f"  model      : {cfg['model']['name']}")
print(f"  n_examples : {cfg['data']['n_source_examples']}")
print(f"  raw_dir    : {cfg['data']['raw_dir']}")

## Step 4 — Download Datasets

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)-8s  %(message)s')

from phase1_dataset.download import download_all
paths = download_all(cfg)
for ds, splits in paths.items():
    for split, p in splits.items():
        print(f'  {ds}/{split} → {p}')

## Step 5 — Generate CoT + Extract Hidden States
> ⏱ **Estimated time**: ~2 hrs for 2000 examples on T4 GPU

In [ ]:
from phase1_dataset.generate_cot import run_generation
cot_path = run_generation(cfg, dry_run=False)
print(f'CoT saved to: {cot_path}')

## Step 6 — Label Hops

In [ ]:
from phase1_dataset.label_hops import run_labeling
labeled_path = run_labeling(cfg)
print(f'Labeled data: {labeled_path}')

# Quick stats
import json
with open(labeled_path) as f:
    examples = [json.loads(l) for l in f]

total_hops = sum(len(e['hops']) for e in examples)
fail_hops  = sum(1 for e in examples for h in e['hops'] if h['label'] == 1)
print(f'Total hops: {total_hops}  |  Failures: {fail_hops} ({100*fail_hops/total_hops:.1f}%)')

## Step 7 — Build Counterfactuals (class balancing)

In [ ]:
from phase1_dataset.counterfactuals import run_counterfactuals
augmented_path = run_counterfactuals(cfg)
print(f'Augmented data: {augmented_path}')

## Step 8 — Split and Write Final JSONL Files

In [ ]:
import random
from phase1_dataset.build_dataset import split_by_id, check_class_balance, check_no_leakage, save_jsonl, sample_hotpotqa
from pathlib import Path

with open(augmented_path) as f:
    all_examples = [json.loads(l) for l in f]

rng = random.Random(cfg.get('seed', 42))
splits = cfg['data']['splits']
train, val, test = split_by_id(all_examples, splits['train'], splits['val'], rng)

# Checks
print('Class balance:')
check_class_balance('train', train)
check_class_balance('val',   val)
check_class_balance('test',  test)
check_no_leakage(train, val, test)

# Write
processed = Path(cfg['data']['processed_dir'])
save_jsonl(train, processed / 'train.jsonl')
save_jsonl(val,   processed / 'val.jsonl')
save_jsonl(test,  processed / 'test.jsonl')

# HotpotQA OOD
hotpot = sample_hotpotqa(cfg, rng)
if hotpot:
    save_jsonl(hotpot, processed / 'hotpotqa_test.jsonl')

print(f'\nDone!  train={len(train)} | val={len(val)} | test={len(test)}')

## Step 9 — Inspect a Sample

In [ ]:
import json, random

with open(processed / 'train.jsonl') as f:
    train_data = [json.loads(l) for l in f]

# Show 3 random examples
for ex in random.sample(train_data, 3):
    print('─' * 60)
    print(f"Q: {ex['question']}")
    print(f"Gold answer: {ex['gold_answer']}")
    print(f"Counterfactual: {ex.get('is_counterfactual', False)}")
    for h in ex['hops']:
        status = '✓' if h['label'] == 0 else '✗'
        print(f"  hop{h['hop_idx']} [{status}] ({h['match_method']}) gold={h['bridging_entity_gold']!r}")
        print(f"         text: {h['text'][:80]}...")
    print(f"  first_fail_hop: {ex['first_fail_hop']}")

## ✅ Phase 1 Complete!

Output files are in `/kaggle/working/data/processed/`.
Download them via **Data → Output** in the Kaggle sidebar.

Next step: **Phase 2 — Train the hop-failure probe** on `train.jsonl` using the hidden states in `/kaggle/working/data/hidden_states/`.